# 07 — Baseline consolidation (B1): retrieval complementarity and the scoring protocol

**Objective.** Consolidate B1 offline, then run the separately approved E8 reader smoke behind an explicit paid-mode gate.
- **A. Retrieval complementarity.**
  - Where does ColQwen2 find annotated evidence that OCR + BM25 misses, and the reverse?
  - Does a simple combination of the two beat ColQwen2 alone at B1's page budget?
  - The answer decides whether B1 retrieves with a fusion or with ColQwen2, keeping BM25 as a first-class second component.
- **B. Scoring protocol v2.**
  - The frozen scorer (S-v1) is kept unchanged.
  - A deterministic second layer and a blind adjudication queue (S-v2) are added on top.
  - The frozen E6 and E7 predictions are scored under both protocols, so later results stay comparable with history.

**Development data only.**
- The 20 documents already used are in scope here: the 6 of E0–E4 and the 14 of E5–E7, with 175 questions.
- The untouched test pool is filtered out at the first step and never read (`experiments/test_pool_ledger.md`).
- Every number in this notebook is a development result.

**Inputs** (the notebook stops with a clear message if one is missing):
- `data/processed/mmlongbench_doc/questions.parquet` and `pages.parquet` (notebook 00);
- `data/processed/mmlongbench_doc/e5/heldout_pages.parquet` (notebook 04);
- `experiments/frozen/e5/` (rankings, per-question retrieval), `e6/` and `e7/` (frozen runs and tables);
- `notebooks/03_evaluation.ipynb` (the frozen scorer cell `f653d232`, compared with the copy written out here).

**Outputs** (git-ignored, in `data/processed/mmlongbench_doc/b1/`):
- `retrieval_complementarity.csv` and `retrieval_summary.json`;
- `scoring_v1_v2.csv` and `scoring_summary.json`;
- `adjudication_queue.csv`, blind, for a human adjudicator, and `adjudication_key.csv`, which un-blinds it.

Nothing under `experiments/frozen/`, `src/visualdoc/` or `AGENTS.md` is written to. The optional E8 smoke writes only to git-ignored `experiments/runs/e8_smoke/`.

**Next step.** E8 is gated in stages: the approved smoke below measures the reader profile; the 354-call full stage remains closed until separately approved.

## Method

### A. Retrieval complementarity

**Metrics.** For a question with annotated evidence pages $E$ and the first $k$ pages $S_k$ of a ranking:
- $\text{Recall@}k = |E \cap S_k| / |E|$;
- $\text{Hit@}k = \mathbb{1}[E \cap S_k \neq \emptyset]$;
- $\text{Complete@}k = \mathbb{1}[E \subseteq S_k]$.

$k = 4$ is B1's page budget, the number of pages E6's reader actually read for 105 of 118 questions; $k = 8$ is shown for reference.

**Rankings.** All are taken from E5's frozen rankings; nothing is re-encoded here.
- **A.** OCR + BM25.
- **V.** ColQwen2 v1.0.
- **H.** Reciprocal-rank fusion (RRF), $\text{score}(d) = \sum_{r \in \{A, V\}} 1/(60 + \text{rank}_r(d))$, as frozen in E5.
- **I.** Interleaving, exploratory: $V_1, A_1, V_2, A_2, \dots$ without repeats. Each retriever gets every second slot.
- **Either.** The per-question maximum of A and V. It is an upper bound for *any* rule that picks one retriever per question, not an implementable system.

**Statistics.** The unit is the question, paired across rankings and clustered in documents.
- Each difference against V is reported with a document-cluster percentile bootstrap: 10,000 draws of documents with replacement, seed 20261125.
- The numbers are descriptive: these questions were already used to choose ColQwen2.

**Decision rule, fixed before any number below was computed.** B1 retrieves with a combination (H or I) instead of V only if both hold:
1. on the 101 held-out evidence questions, its Complete@4 exceeds V's by at least 0.05, with the 95 % cluster interval above 0;
2. it is not worse than V in the core-scope subgroup.

Even then, the combination becomes a *candidate for a controlled test*, not an immediate change, because these are development questions. Otherwise V stays primary, and BM25 keeps four first-class roles:
- the fallback when no visual index exists;
- the text layer used for grounding checks;
- a ranking logged beside V's;
- the comparator.

### B. Scoring protocol v2

**S-v1, frozen.** The MMLongBench-Doc rule-based scorer as re-implemented in notebook 03 (cell `f653d232`), written out below unchanged and checked function by function against that cell. A reply is correct when score > 0, strict when score = 1. Every historical number and verdict uses S-v1 and stays as it is.

**S-v2.** A deterministic layer classifies each reply as **correct**, **wrong** or **gray**. Gray replies go to blind human adjudication. Until they are labelled, S-v2 is reported as an interval: gray counted wrong (lower) and gray counted correct (upper).

| Reference format | Rule, in order |
|---|---|
| `None` (unanswerable) | correct only for the exact abstention "Not answerable" (as S-v1) |
| any other format | "Not answerable" is wrong (a false abstention); S-v1 = 1 is correct |
| `Int`, `Float` | Parse one number from each side, after removing currency signs and thousands separators, with `%` and scale words (thousand, million, billion…). Equal is correct: exactly for `Int`; within 1 % for `Float`, including the official ×100 / ÷100 percentage variants. Equal up to a factor 1,000ⁿ is gray (unit scale ambiguous). Several numbers in the reply, one of them equal, is gray. Otherwise wrong. |
| `Str` | Both sides a single equal number: correct; equal up to scale: gray. Equal after case and punctuation normalisation: correct. 0 < S-v1 < 1 (ANLS partial credit): gray. The normalised reference is a phrase inside the reply: gray. The reply (≥ 2 words) is a phrase inside the reference: gray. Otherwise wrong. |
| `List` | The same set of items after normalisation (numbers compared as numbers) is correct. Gray when every reference item is matched by a reply item (equal, one a phrase of the other, or ANLS > 0.5), with or without extra items. Wrong when a reference item is missing, because the benchmark asks for the whole list. |
| any format | A reference the frozen E6 error audit labelled an *annotation issue* (`mmlb-0689`, `mmlb-1013`) is gray unless S-v1 = 1. |

*Revision before the protocol is frozen (2026-11-25).* The first execution sent every list sharing any item with the reference to adjudication: 22 of 37 items, nearly all missing reference items. The list rule above replaced it; no other rule changed.

**Why a gray zone instead of a new automatic rule.** String similarity cannot tell "BBA" from a wrong programme name that shares most of its characters, or a correct sentence from one that negates the answer. The frozen E7 analysis found both errors in S-v1:
- `mmlb-0998`: a wrong reply scored 0.951;
- `mmlb-0684`, `mmlb-0593`: correct but verbose replies scored 0.

A human resolves the few ambiguous replies, blind to the system that produced them.

**Adjudication rubric**, applied blind: the adjudicator sees the question, document, evidence pages, reference and reply, never the arm.
- **Correct:** the reply gives the reference's answer (the same entity, value or set; units consistent). Added explanation is allowed if it does not contradict it.
- **Wrong:** a different answer; several alternatives without committing; or a final answer that differs from the reference, even if the reference appears in the text.
- **Unsure:** the reference itself looks wrong or ambiguous. These go to the "debatable reference" list and are reported separately.

In [1]:
# Setup: paths, inputs and constants. Sections A and B are offline; section C defaults to replay and creates a client only in approved smoke mode.
from collections import Counter
from math import isclose
from pathlib import Path
import ast
import hashlib
import inspect
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc"
OUT_DIR = PROCESSED_DIR / "b1"
FROZEN_DIR = PROJECT_ROOT / "experiments" / "frozen"
E6_RUN = FROZEN_DIR / "e6" / "run_20261123T200722253880Z_d94e85f1_full"
E7_RUN = FROZEN_DIR / "e7" / "run_20261124T160111456175Z_b3d0b5c0_full"
E6_SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "e6_visual_retrieval_answering_v1.json"
NB03_PATH = PROJECT_ROOT / "notebooks" / "03_evaluation.ipynb"

REQUIRED_INPUTS = {
    PROCESSED_DIR / "questions.parquet": "run 00_document_processing",
    PROCESSED_DIR / "pages.parquet": "run 00_document_processing",
    PROCESSED_DIR / "e5" / "heldout_pages.parquet": "run 04_visual_retrieval",
    E6_SPEC_PATH: "the E6 specification is tracked in git: restore it",
    NB03_PATH: "notebook 03 is tracked in git: restore it",
    **{FROZEN_DIR / name / "SHA256SUMS": f"the frozen {name.upper()} result is tracked in git: restore experiments/frozen/{name}" for name in ("e5", "e6", "e7")},
}
missing = [f"{path.relative_to(PROJECT_ROOT)} ({fix})" for path, fix in REQUIRED_INPUTS.items() if not path.is_file()]
assert not missing, "Notebook 07 inputs missing:\n  " + "\n  ".join(missing)
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20261125  # cluster bootstrap and the order of the adjudication queue
DRAWS = 10_000
B1_K = 4  # B1's page budget
REPORT_TYPES = {"Financial report", "Research report / Introduction", "Administration/Industry file"}
NATIVE_WORDS_MIN = 30  # notebook 00's OCR threshold: a page with fewer native words is OCR'd
BORN_DIGITAL_MIN_SHARE = 0.80  # a document is born-digital when at least 80 % of its pages have a usable text layer
NOT_ANSWERABLE = "Not answerable"
print(f"Project root: {PROJECT_ROOT.name}. Outputs: {OUT_DIR.relative_to(PROJECT_ROOT)}. Sections A-B offline; E8 defaults to replay.")

Project root: Multimodal-Document-Intelligence. Outputs: data/processed/mmlongbench_doc/b1. Sections A-B offline; E8 defaults to replay.


In [2]:
# The frozen E5, E6 and E7 packages are read-only inputs: every file must equal its SHA256SUMS.
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def verify_sha256sums(folder):
    entries = [line.split(maxsplit=1) for line in (folder / "SHA256SUMS").read_text().splitlines() if line.strip()]
    return {name: (folder / name).is_file() and sha256_bytes((folder / name).read_bytes()) == digest for digest, name in entries}


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


frozen_checks = {name: verify_sha256sums(FROZEN_DIR / name) for name in ("e5", "e6", "e7")}
for name, checks in frozen_checks.items():
    assert checks and all(checks.values()), f"experiments/frozen/{name} differs from its SHA256SUMS: {[k for k, ok in checks.items() if not ok]}"
print("Frozen inputs equal their SHA256SUMS: " + ", ".join(f"{name} ({len(c)} files)" for name, c in frozen_checks.items()))

Frozen inputs equal their SHA256SUMS: e5 (11 files), e6 (31 files), e7 (38 files)


In [3]:
# The development population: the 20 documents already used. The test pool is dropped here, before anything else is read.
questions = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
DEV6 = sorted(questions.loc[questions["in_subset"], "doc_id"].unique())
HELDOUT14 = sorted(json.loads(E6_SPEC_PATH.read_text())["population"]["documents"])
USED_DOCS = set(DEV6) | set(HELDOUT14)
assert len(DEV6) == 6 and len(HELDOUT14) == 14 and not set(DEV6) & set(HELDOUT14)
dev = questions[questions["doc_id"].isin(USED_DOCS)].set_index("qid").copy()
del questions  # nothing below can read a test-pool row
dev["evidence_pages"] = dev["evidence_pages"].map(lambda pages: [int(p) for p in pages])
dev["evidence_sources"] = dev["evidence_sources"].map(list)
dev["set"] = np.where(dev["doc_id"].isin(DEV6), "dev6", "held-out14")
assert len(dev) == 175 and int(dev["answerable"].sum()) == 142 and dev["doc_id"].nunique() == 20

# Born-digital: the share of a document's pages whose native text layer has at least 30 words (notebook 00's OCR rule).
pages = pd.concat([pd.read_parquet(PROCESSED_DIR / "pages.parquet", columns=["doc_id", "page", "native_words"]),
                   pd.read_parquet(PROCESSED_DIR / "e5" / "heldout_pages.parquet", columns=["doc_id", "page", "native_words"])])
assert set(pages["doc_id"]) == USED_DOCS
documents = pages.groupby("doc_id").agg(pages=("page", "size"), native_share=("native_words", lambda s: float((s >= NATIVE_WORDS_MIN).mean())))
documents["doc_type"] = dev.groupby("doc_id")["doc_type"].first()
documents["set"] = dev.groupby("doc_id")["set"].first()
documents["questions"] = dev.groupby("doc_id").size()
documents["born_digital"] = documents["native_share"] >= BORN_DIGITAL_MIN_SHARE
documents["core_scope"] = documents["born_digital"] & documents["doc_type"].isin(REPORT_TYPES)
dev["core_scope"] = dev["doc_id"].map(documents["core_scope"])
dev["span"] = dev["evidence_pages"].map(lambda e: "none" if not e else "1 page" if len(e) == 1 else "2-3 pages" if len(e) <= 3 else "4+ pages")
display(documents.sort_values(["core_scope", "doc_type"], ascending=[False, True]).round(2))

# How sensitive is the core-scope set to the born-digital threshold?
sensitivity = {thr: sorted(documents.index[(documents["native_share"] >= thr) & documents["doc_type"].isin(REPORT_TYPES)]) for thr in (0.5, 0.6, 0.7, 0.8, 0.84)}
CORE_DOCS = sorted(documents.index[documents["core_scope"]])
print(f"Core scope (born-digital report-type documents): {len(CORE_DOCS)} of 20 documents, {int(dev['core_scope'].sum())} of 175 questions "
      f"({int((dev['core_scope'] & dev['answerable']).sum())} answerable). The set is the same for every threshold from 0.5 to 0.84: "
      f"{all(v == CORE_DOCS for v in sensitivity.values())}.")
assert all(v == CORE_DOCS for v in sensitivity.values())

,pages,native_share,doc_type,set,questions,born_digital,core_scope
doc_id,,,,,,,
e639029d16094ea71d964e2fb953952b.pdf,20,0.95,Administration/Industry file,held-out14,7,True,True
AMAZON_2017_10K.pdf,85,0.99,Financial report,held-out14,15,True,True
COSTCO_2021_10K.pdf,76,1.00,Financial report,held-out14,16,True,True
11-21-16-Updated-Post-Election-Release.pdf,45,1.00,Research report / Introduction,held-out14,8,True,True
PG_2020.05.21_International-Cooperation-COVID_FINAL.pdf,26,0.96,Research report / Introduction,dev6,11,True,True
efd88e41c5f2606c57929cac6c1c0605.pdf,20,0.85,Research report / Introduction,held-out14,7,True,True
2210.02442v1.pdf,24,1.00,Academic paper,dev6,9,True,False
2310.05634v2.pdf,24,1.00,Academic paper,held-out14,7,True,False
2312.10997v5.pdf,21,1.00,Academic paper,held-out14,7,True,False


Core scope (born-digital report-type documents): 6 of 20 documents, 64 of 175 questions (57 answerable). The set is the same for every threshold from 0.5 to 0.84: True.


## A. Retrieval complementarity

**Step 1.** Every ranking is truncated to $k$ pages, and the three evidence metrics are computed per question. They are checked first against E5's frozen per-question table.

**Step 2.** Then come four tables:
- the A/V contingency: who finds what the other misses;
- recall by stratum;
- the paired comparison of each combination against V;
- the answer-level view from E6, where both rankings were actually read by the same reader.

In [4]:
# Per-question evidence metrics at k = 4 (B1) and k = 8, for A, V, H (frozen E5 rankings) and the interleave I.
rankings = pd.read_parquet(FROZEN_DIR / "e5" / "tables" / "rankings.parquet")
RANKING = {(r.qid, r.arm): [int(p) for p in r.pages] for r in rankings.itertuples()}
assert {qid for qid, _ in RANKING} == set(dev.index) and {arm for _, arm in RANKING} == {"A", "V", "H"}
EVIDENCE_QIDS = [q for q in dev.index if dev.loc[q, "answerable"] and dev.loc[q, "evidence_pages"]]
assert len(EVIDENCE_QIDS) == 141


def interleave(first, second, k):
    """first[0], second[0], first[1], second[1], ... without repeats, until k pages."""
    pages = []
    for pair in zip(first, second):
        for page in pair:
            if page not in pages:
                pages.append(page)
            if len(pages) == k:
                return pages
    return pages


def evidence_metrics(evidence, supplied):
    found = set(evidence) & set(supplied)
    return {"recall": len(found) / len(evidence), "hit": float(bool(found)), "complete": float(set(evidence) <= set(supplied))}


def first_rank(evidence, ranking):
    return min((ranking.index(p) + 1 for p in evidence if p in ranking), default=None)


rows = []
for qid in EVIDENCE_QIDS:
    evidence = dev.loc[qid, "evidence_pages"]
    for k in (B1_K, 8):
        supplied = {"A": RANKING[(qid, "A")][:k], "V": RANKING[(qid, "V")][:k], "H": RANKING[(qid, "H")][:k],
                    "I": interleave(RANKING[(qid, "V")], RANKING[(qid, "A")], k)}
        row = {"qid": qid, "k": k, "first_rank_A": first_rank(evidence, RANKING[(qid, "A")]), "first_rank_V": first_rank(evidence, RANKING[(qid, "V")])}
        for arm, pages_k in supplied.items():
            assert len(pages_k) == min(k, len(RANKING[(qid, "V")]))
            row.update({f"{metric}_{arm}": value for metric, value in evidence_metrics(evidence, pages_k).items()})
        row.update({f"{metric}_either": max(row[f"{metric}_A"], row[f"{metric}_V"]) for metric in ("recall", "hit", "complete")})
        rows.append(row)
retrieval = pd.DataFrame(rows).merge(dev[["doc_id", "doc_type", "set", "core_scope", "span", "evidence_sources"]], left_on="qid", right_index=True)

# Check against E5's frozen per-question table: raw Recall@4 and Hit@4 of A, V and H are reproduced for all 141 questions.
e5_table = pd.read_csv(FROZEN_DIR / "e5" / "tables" / "per_question.csv").set_index("qid")
assert set(e5_table.index) == set(EVIDENCE_QIDS)
k4 = retrieval[retrieval["k"] == B1_K].set_index("qid")
for arm in "AVH":
    assert np.allclose(k4[f"recall_{arm}"], e5_table.loc[k4.index, f"recall@4_{arm}"]) and np.allclose(k4[f"hit_{arm}"], e5_table.loc[k4.index, f"hit@4_{arm}"]), arm
retrieval = retrieval.merge(e5_table[["evidence_on_ocr_page"]], left_on="qid", right_index=True)
k4 = retrieval[retrieval["k"] == B1_K]
print(f"{len(EVIDENCE_QIDS)} evidence questions (40 dev6 + 101 held-out); raw Recall@4 and Hit@4 of A, V and H equal E5's frozen table on all of them.")

141 evidence questions (40 dev6 + 101 held-out); raw Recall@4 and Hit@4 of A, V and H equal E5's frozen table on all of them.


In [5]:
# Who finds what the other misses, at B1's budget (k = 4): complete evidence and any evidence, A (BM25) against V (ColQwen2).
def contingency(frame, metric):
    a, v = frame[f"{metric}_A"].astype(bool), frame[f"{metric}_V"].astype(bool)
    return {"questions": len(frame), "both": int((a & v).sum()), "BM25 only": int((a & ~v).sum()),
            "ColQwen2 only": int((~a & v).sum()), "neither": int((~a & ~v).sum())}


GROUPS = {"all 141": k4, "held-out 101": k4[k4["set"] == "held-out14"], "dev6 40": k4[k4["set"] == "dev6"],
          "core scope": k4[k4["core_scope"]], "other documents": k4[~k4["core_scope"]]}
contingency_table = pd.DataFrame({(group, metric): contingency(frame, metric) for group, frame in GROUPS.items()
                                  for metric in ("complete", "hit")}).T
display(contingency_table)

# Where one retriever misses complete evidence, how far down the other's ranking is the first evidence page?
misses = {"BM25 only (V incomplete, A complete)": k4[(k4["complete_A"] == 1) & (k4["complete_V"] == 0)],
          "ColQwen2 only (A incomplete, V complete)": k4[(k4["complete_A"] == 0) & (k4["complete_V"] == 1)]}
display(pd.DataFrame({label: {"questions": len(frame), "core scope": int(frame["core_scope"].sum()),
                              "by type": dict(Counter(frame["doc_type"])), "median first evidence rank in the losing ranking":
                              float(frame["first_rank_V" if label.startswith("BM25") else "first_rank_A"].median()) if len(frame) else None}
                      for label, frame in misses.items()}).T)

questions  both  BM25 only  ColQwen2 only  neither
all 141         complete        141    60          4             32       45
                hit             141    93          4             31       13
held-out 101    complete        101    39          4             23       35
                hit             101    65          3             21       12
dev6 40         complete         40    21          0              9       10
                hit              40    28          1             10        1
core scope      complete         57    25          2              6       24
                hit              57    38          3              6       10
other documents complete         84    35          2             26       21
                hit              84    55          1             25        3

,questions,core scope,by type,median first evidence rank in the losing ranking
"BM25 only (V incomplete, A complete)",4,2,"{'Tutorial/Workshop': 1, 'Academic paper': 1, 'Administration/Indu...",3.0
"ColQwen2 only (A incomplete, V complete)",32,6,"{'Research report / Introduction': 7, 'Tutorial/Workshop': 5, 'Aca...",7.5


In [6]:
# Mean Recall@4 by stratum for every ranking, and the per-question ceiling "either" (a perfect choice between A and V).
def recall_by(frame, column):
    table = frame.groupby(column)[[f"recall_{arm}" for arm in ("A", "V", "H", "I", "either")]].mean()
    table.insert(0, "questions", frame.groupby(column).size())
    return table.round(3)


by_source = k4.explode("evidence_sources").rename(columns={"evidence_sources": "evidence source"})
for title, table in {"core scope": recall_by(k4, "core_scope"), "set": recall_by(k4, "set"), "evidence span": recall_by(k4, "span"),
                     "evidence on an OCR'd page": recall_by(k4, "evidence_on_ocr_page"),
                     "evidence source (a question counts once per source)": recall_by(by_source, "evidence source"),
                     "document type": recall_by(k4, "doc_type")}.items():
    print(f"Recall@4 by {title}")
    display(table)

Recall@4 by core scope


,questions,recall_A,recall_V,recall_H,recall_I,recall_either
core_scope,,,,,,
False,84,0.556,0.829,0.675,0.756,0.849
True,57,0.581,0.656,0.649,0.628,0.700


Recall@4 by set


,questions,recall_A,recall_V,recall_H,recall_I,recall_either
set,,,,,,
dev6,40,0.629,0.843,0.711,0.766,0.851
held-out14,101,0.541,0.726,0.646,0.680,0.764


Recall@4 by evidence span


,questions,recall_A,recall_V,recall_H,recall_I,recall_either
span,,,,,,
1 page,77,0.701,0.909,0.805,0.870,0.935
2-3 pages,55,0.439,0.597,0.524,0.539,0.627
4+ pages,9,0.180,0.467,0.318,0.293,0.523


... (2 outputs left out)

Recall@4 by evidence source (a question counts once per source)


,questions,recall_A,recall_V,recall_H,recall_I,recall_either
evidence source,,,,,,
Chart,27,0.785,0.865,0.802,0.832,0.865
Figure,42,0.480,0.854,0.644,0.742,0.885
Generalized-text (Layout),18,0.380,0.788,0.567,0.657,0.788
Pure-text (Plain-text),42,0.627,0.718,0.732,0.661,0.798
Table,43,0.483,0.609,0.587,0.583,0.620


Recall@4 by document type


,questions,recall_A,recall_V,recall_H,recall_I,recall_either
doc_type,,,,,,
Academic paper,18,0.463,0.778,0.611,0.694,0.824
Administration/Industry file,12,0.625,0.750,0.750,0.708,0.917
Brochure,8,0.562,0.875,0.625,0.875,0.875
Financial report,31,0.508,0.548,0.573,0.540,0.565
Guidebook,20,0.600,0.792,0.683,0.758,0.808
Research report / Introduction,38,0.663,0.854,0.754,0.785,0.854
Tutorial/Workshop,14,0.464,0.839,0.616,0.682,0.875


In [7]:
# Each combination against V, paired by question, with a document-cluster bootstrap (descriptive: development data).
def cluster_difference(frame, metric, arm, base="V", seed=SEED, draws=DRAWS):
    diff = (frame[f"{metric}_{arm}"] - frame[f"{metric}_{base}"]).to_numpy()
    docs = frame["doc_id"].to_numpy()
    order = sorted(set(docs))
    sums = np.array([diff[docs == d].sum() for d in order])
    counts = np.array([(docs == d).sum() for d in order])
    picks = np.random.default_rng(seed).integers(0, len(order), size=(draws, len(order)))
    boot = sums[picks].sum(axis=1) / counts[picks].sum(axis=1)
    low, high = np.percentile(boot, [2.5, 97.5])
    return {"questions": len(diff), "documents": len(order), "delta": round(float(diff.mean()), 3),
            "ci95": f"[{low:+.3f}, {high:+.3f}]", "ci_low": float(low), "better": int((diff > 0).sum()), "worse": int((diff < 0).sum())}


comparison_rows = {}
for k in (B1_K, 8):
    at_k = retrieval[retrieval["k"] == k]
    for group, frame in {"held-out 101": at_k[at_k["set"] == "held-out14"], "core scope": at_k[at_k["core_scope"]],
                         "other documents": at_k[~at_k["core_scope"]]}.items():
        for arm in ("A", "H", "I", "either"):
            for metric in ("complete", "recall"):
                comparison_rows[(k, group, f"{arm} - V", metric)] = cluster_difference(frame, metric, arm)
comparison = pd.DataFrame(comparison_rows).T
comparison.index.names = ["k", "population", "comparison", "metric"]
display(comparison.drop(columns="ci_low"))

# The decision rule stated in the Method section, applied to H and I at k = 4.
decision = {}
for arm in ("H", "I"):
    held = comparison.loc[(B1_K, "held-out 101", f"{arm} - V", "complete")]
    core = comparison.loc[(B1_K, "core scope", f"{arm} - V", "complete")]
    decision[arm] = {"Complete@4 gain >= 0.05 on held-out": held["delta"] >= 0.05, "95 % interval above 0": held["ci_low"] > 0,
                     "not worse in the core scope": core["delta"] >= 0, "adopt as a candidate": None}
    decision[arm]["adopt as a candidate"] = all(v for key, v in decision[arm].items() if key != "adopt as a candidate")
display(pd.DataFrame(decision))
RETRIEVAL_DECISION = "a combination becomes a candidate for a controlled test" if any(d["adopt as a candidate"] for d in decision.values()) \
    else "ColQwen2 (V) stays B1's primary retriever; no fusion is adopted"
print("Retrieval decision (pre-stated rule):", RETRIEVAL_DECISION)

questions documents  delta              ci95 better worse
k population      comparison metric                                                            
4 held-out 101    A - V      complete       101        14 -0.188  [-0.305, -0.097]      4    23
                             recall         101        14 -0.185  [-0.292, -0.102]      6    30
                  H - V      complete       101        14 -0.079  [-0.167, -0.016]      4    12
                             recall         101        14  -0.08  [-0.164, -0.020]      7    14
                  I - V      complete       101        14 -0.059  [-0.133, -0.009]      1     7
                             recall         101        14 -0.046  [-0.093, -0.013]      2    12
                  either - V complete       101        14   0.04  [+0.000, +0.095]      4     0
                             recall         101        14  0.038  [+0.009, +0.085]      6     0
  core scope      A - V      complete        57         6  -0.07  [-0.213, +0.000]      2     6
                             recall          57         6 -0.075  [-0.207, -0.010]      3    10
                  H - V      complete        57         6    0.0  [-0.064, +0.039]      3     3
                             recall          57         6 -0.007  [-0.069, +0.023]      4     5
                  I - V      complete        57         6 -0.018  [-0.065, +0.000]      0     1
                             recall          57         6 -0.028  [-0.082, -0.003]      0     4
                  either - V complete        57         6  0.035  [+0.000, +0.128]      2     0
                             recall          57         6  0.044  [+0.000, +0.138]      3     0
  other documents A - V      complete        84        14 -0.286  [-0.365, -0.205]      2    26
                             recall          84        14 -0.273  [-0.343, -0.206]      4    34
                  H - V      complete        84        14 -0.155  [-0.228, -0.093]      1    14
                             recall          84        14 -0.154  [-0.227, -0.091]      3    18
                  I - V      complete        84        14 -0.095  [-0.175, -0.026]      1     9
                             recall          84        14 -0.073  [-0.122, -0.032]      2    15
                  either - V complete        84        14  0.024  [+0.000, +0.058]      2     0
                             recall          84        14   0.02  [+0.004, +0.038]      4     0
8 held-out 101    A - V      complete       101        14 -0.139  [-0.293, -0.031]      4    18
                             recall         101        14 -0.151  [-0.281, -0.052]      7    26
                  H - V      complete       101        14  -0.03  [-0.104, +0.027]      4     7
                             recall         101        14 -0.018  [-0.068, +0.018]      6     9
                  I - V      complete       101        14  -0.01  [-0.070, +0.038]      2     3
                             recall         101        14 -0.007  [-0.034, +0.022]      2     5
                  either - V complete       101        14   0.04  [+0.011, +0.067]      4     0
                             recall         101        14   0.04  [+0.005, +0.080]      7     0
  core scope      A - V      complete        57         6    0.0  [-0.105, +0.052]      3     3
                             recall          57         6  -0.02  [-0.125, +0.042]      6     8
                  H - V      complete        57         6  0.018  [+0.000, +0.044]      2     1
                             recall          57         6  0.013  [-0.018, +0.032]      4     3
                  I - V      complete        57         6 -0.018  [-0.120, +0.055]      1     2
                             recall          57         6 -0.004  [-0.059, +0.055]      1     3
                  either - V complete        57         6  0.053  [+0.020, +0.089]      3     0
                             recall          57         6  0.067  [+0.010, +0.125]      6     0
  othe

,H,I
Complete@4 gain >= 0.05 on held-out,False,False
95 % interval above 0,False,False
not worse in the core scope,True,False
adopt as a candidate,False,False


Retrieval decision (pre-stated rule): ColQwen2 (V) stays B1's primary retriever; no fusion is adopted


In [8]:
# The answer-level view: in E6 the same reader read A's and V's pages. Which retriever's pages led to a correct answer?
e6_table = pd.read_csv(FROZEN_DIR / "e6" / "tables" / "per_question.csv").set_index("qid")
e6_answerable = e6_table[e6_table["answerable"]].join(dev[["core_scope"]])
answers = {}
for group, frame in {"all 102": e6_answerable, "core scope": e6_answerable[e6_answerable["core_scope"]],
                     "other documents": e6_answerable[~e6_answerable["core_scope"]]}.items():
    a, v = frame["correct_A"].astype(bool), frame["correct_V"].astype(bool)
    answers[group] = {"questions": len(frame), "both correct": int((a & v).sum()), "only with BM25 pages": int((a & ~v).sum()),
                      "only with ColQwen2 pages": int((~a & v).sum()), "neither": int((~a & ~v).sum())}
display(pd.DataFrame(answers).T)
assert answers["all 102"]["only with BM25 pages"] == 0 and answers["all 102"]["only with ColQwen2 pages"] == 13  # frozen E6: 13 fixed, 0 broken

,questions,both correct,only with BM25 pages,only with ColQwen2 pages,neither
all 102,102,25,0,13,64
core scope,50,16,0,5,29
other documents,52,9,0,8,35


In [9]:
# Save the per-question table and the summary (no question, answer or page text).
retrieval.drop(columns=["evidence_sources"]).assign(evidence_sources=retrieval["evidence_sources"].map("|".join)).to_csv(
    OUT_DIR / "retrieval_complementarity.csv", index=False)
retrieval_summary = {
    "population": "development: 141 evidence questions in 20 documents (40 dev6 + 101 held-out); core scope = born-digital report-type documents",
    "core_scope_documents": CORE_DOCS, "k": [B1_K, 8],
    "contingency_k4": {f"{g} / {m}": v for (g, m), v in contingency_table.to_dict(orient="index").items()},
    "comparisons_vs_V": {" | ".join(map(str, key)): {k: v for k, v in value.items() if k != "ci_low"} for key, value in comparison.to_dict(orient="index").items()},
    "answer_level_e6": answers, "decision_rule_outcome": {arm: {k: bool(v) for k, v in d.items()} for arm, d in decision.items()},
    "decision": RETRIEVAL_DECISION,
}
(OUT_DIR / "retrieval_summary.json").write_text(json.dumps(retrieval_summary, indent=1, default=str))
print(f"Saved {OUT_DIR.relative_to(PROJECT_ROOT)}/retrieval_complementarity.csv and retrieval_summary.json")

Saved data/processed/mmlongbench_doc/b1/retrieval_complementarity.csv and retrieval_summary.json


## B. Scoring protocol v2

1. **S-v1 is written out unchanged.** It is checked, function by function, against notebook 03's frozen cell `f653d232`, against its 17 hand-traced cases, and against every frozen E6 and E7 score.
2. **The S-v2 layer is defined.** It is tested on invented examples, including cases it cannot settle.
3. **The frozen predictions are scored under both protocols:** E6 A and V, and E7 V0, X4 and X8, on the 118 development questions.
4. **The gray replies become a blind adjudication queue.**

In [10]:
# S-v1: the frozen MMLongBench-Doc scorer (notebook 03, cell f653d232), written out unchanged. Correct = score > 0; strict = score == 1.
def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            new_distances.append(distances[i1] if c1 == c2 else 1 + min(distances[i1], distances[i1 + 1], new_distances[-1]))
        distances = new_distances
    return distances[-1]


def anls_compute(groundtruth, prediction, threshold=0.5):
    length = max(len(groundtruth.upper()), len(prediction.upper()))
    anls = 1.0 - (0.0 if length == 0 else levenshtein_distance(groundtruth, prediction) / length)
    return 0.0 if anls <= threshold else anls


def is_float_equal(reference, prediction, include_percentage=False, is_close=False):
    def get_precision(value):
        text = str(value)
        return len(text.split(".")[-1]) if "." in text else 3

    reference = float(str(reference).strip().rstrip("%").strip())
    try:
        prediction = float(str(prediction).strip().rstrip("%").strip())
    except (TypeError, ValueError):
        return False
    for item in ([reference / 100, reference, reference * 100] if include_percentage else [reference]):
        if is_close and isclose(item, prediction, rel_tol=0.01):
            return True
        precision = max(min(get_precision(prediction), get_precision(item)), 2)
        if round(prediction, precision) == round(item, precision):
            return True
    return False


def get_clean_string(s):
    s = str(s).lower().strip()
    # Official quirk kept: the code calls s.rstrip("mile") etc. without assigning the result, so unit suffixes stay.
    s = re.sub(r"\s*\([^)]*\)", "", s).strip()  # remove parenthesised text
    s = re.sub(r"^['\"]|['\"]$", "", s).strip()  # remove surrounding quotes
    s = s.strip().lstrip("$").strip()
    return s.strip().rstrip("%").strip()


EXACT_MATCH_PATTERNS = [
    r"\b\d+(-\d+|\s\d+)?\b",  # phone-like numbers
    r"\b\d{4}[-\s]\d{2}[-\s]\d{2}\b",  # YYYY-MM-DD
    r"\b\d{4}[-\s]\d{2}\b",  # YYYY-MM
    r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # e-mail address
]


def is_exact_match(s):
    return ("https://" in s or s.endswith(".py") or s.endswith("ipynb") or s.startswith("page") or "a.m." in s or "p.m." in s
            or any(re.fullmatch(pattern, s) for pattern in EXACT_MATCH_PATTERNS))


def isfloat(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def as_list(value):
    if isinstance(value, str) and value.startswith("["):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass  # deviation: the official eval() would raise here
    return value if isinstance(value, list) else [value]


def eval_score(gt, pred, answer_type):
    if answer_type == "Int":
        try:
            gt, pred = int(gt), int(float(pred))
        except Exception:  # the official code uses a bare except
            pred = ""
        score = gt == pred
    elif answer_type == "Float":
        try:
            gt, pred = float(get_clean_string(str(gt))), float(get_clean_string(str(pred)))
        except Exception:
            pred = ""
        score = is_float_equal(gt, pred, include_percentage=True, is_close=True)
    elif answer_type in ("Str", "None"):
        gt, pred = get_clean_string(gt), get_clean_string(pred)
        score = gt == pred if is_exact_match(gt) else anls_compute(gt, pred)
    else:  # List
        gt, pred = as_list(gt), as_list(pred)
        if len(gt) != len(pred):
            score = 0.0
        else:
            gt, pred = sorted(get_clean_string(i) for i in gt), sorted(get_clean_string(i) for i in pred)
            score = "-".join(gt) == "-".join(pred) if isfloat(gt[0]) or is_exact_match(gt[0]) else min(anls_compute(g, p) for g, p in zip(gt, pred))
    return float(score)


# 1. Unchanged: every function above has the same source as the frozen cell, and so does the pattern list.
frozen_cell = next("".join(c["source"]) for c in json.loads(NB03_PATH.read_text())["cells"] if c.get("id") == "f653d232")
frozen_tree = ast.parse(frozen_cell)
frozen_functions = {node.name: ast.get_source_segment(frozen_cell, node) for node in frozen_tree.body if isinstance(node, ast.FunctionDef)}
SCORER_FUNCTIONS = ["levenshtein_distance", "anls_compute", "is_float_equal", "get_clean_string", "is_exact_match", "isfloat", "as_list", "eval_score"]
assert set(SCORER_FUNCTIONS) <= set(frozen_functions)
assert all(inspect.getsource(globals()[name]).strip() == frozen_functions[name].strip() for name in SCORER_FUNCTIONS)
frozen_patterns = next(ast.literal_eval(node.value) for node in frozen_tree.body
                       if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", None) == "EXACT_MATCH_PATTERNS")
assert frozen_patterns == EXACT_MATCH_PATTERNS

# 2. The 17 hand-traced cases of notebook 03.
SCORING_CASES = [  # reference, prediction, format, expected
    ("Less well-off", "less well off", "Str", 1 - 1 / 13), ("Pew Research Center", "Pew Research Centre", "Str", 1 - 2 / 19),
    ("Latinos", "Asian Americans", "Str", 0.0), ("Not answerable", "Not answerable", "None", 1.0), ("Not answerable", "The report does not say", "None", 0.0),
    ("12", "12.0", "Int", 1.0), ("12", "twelve", "Int", 0.0), ("18.29%", "18.3", "Float", 1.0), ("0.1", "10%", "Float", 1.0), ("51%", "49", "Float", 0.0),
    ("['Circle', 'Rectangle']", "['rectangle', 'circle']", "List", 1.0), ("['White', '10%']", "['White']", "List", 0.0),
    ("['2014', '2016']", "['2016', '2014']", "List", 1.0), ("https://www.example.com", "www.example.com", "Str", 0.0), ("2016-06", "2016-06", "Str", 1.0),
    ("$1,200 (approx.)", "1,200", "Str", 1.0), ("5 miles", "5", "Str", 0.0)]
assert len(SCORING_CASES) == 17 and all(isclose(eval_score(r, p, f), e, abs_tol=1e-9) for r, p, f, e in SCORING_CASES)
print(f"S-v1: {len(SCORER_FUNCTIONS)} functions and the pattern list equal notebook 03's frozen cell f653d232; {len(SCORING_CASES)} hand-traced cases pass.")

S-v1: 8 functions and the pattern list equal notebook 03's frozen cell f653d232; 17 hand-traced cases pass.


In [11]:
# S-v2: a deterministic layer over S-v1 that returns (class, reason) with class in {correct, wrong, gray}. Gray goes to blind adjudication.
SCALE_WORDS = {"thousand": 1e3, "k": 1e3, "million": 1e6, "millions": 1e6, "mn": 1e6, "billion": 1e9, "billions": 1e9, "bn": 1e9, "trillion": 1e12}
NUMBER_PATTERN = re.compile(r"(\()?(-?\d[\d,]*(?:\.\d+)?)(\))?\s*(%|(?:percent|thousand|millions?|billions?|trillion|bn|mn|k)\b)?")
NUMERIC_TEXT_RESIDUE = re.compile(r"us\$|usd|eur|gbp|percent|thousand|millions?|billions?|trillion|\bbn\b|\bmn\b|\bk\b|[$€£%(),.\-\s\d]")
POWERS_OF_1000 = (1e-9, 1e-6, 1e-3, 1e3, 1e6, 1e9)  # "m" and "b" are not scale words: metres, bytes


def numbers_in(value):
    """Every number in a reply or reference, after removing currency signs; a scale word multiplies, '(x)' is negative."""
    if isinstance(value, bool):
        return []
    if isinstance(value, (int, float)):
        return [float(value)]
    text = re.sub(r"us\$|usd|eur|gbp|[$€£]", " ", str(value).lower())
    found = []
    for open_paren, digits, close_paren, unit in NUMBER_PATTERN.findall(text):
        number = float(digits.replace(",", ""))
        found.append((-number if open_paren and close_paren else number) * SCALE_WORDS.get(unit, 1.0))
    return found


def numeric_text(value):
    """True when a value is one number with at most currency, %, separators and a scale word around it."""
    if isinstance(value, bool):
        return False
    if isinstance(value, (int, float)):
        return True
    return NUMERIC_TEXT_RESIDUE.sub("", str(value).lower()) == "" and len(numbers_in(value)) == 1


def same_value(reference, prediction, answer_format):
    # Float: within 1 %, including the official x100 / /100 percentage variants; Int: exactly equal.
    if answer_format == "Float":
        return any(isclose(candidate, prediction, rel_tol=0.01, abs_tol=1e-9) for candidate in (reference, reference / 100, reference * 100))
    return isclose(reference, prediction, rel_tol=0.0, abs_tol=1e-9)


def numeric_class(reference, prediction, answer_format):
    """correct / scale / among / wrong / None (the reference is not one number)."""
    ref, pred = numbers_in(reference), numbers_in(prediction)
    if len(ref) != 1 or not pred:
        return None if len(ref) != 1 else "wrong"
    if len(pred) > 1:
        return "among" if any(same_value(ref[0], p, answer_format) for p in pred) else "wrong"
    if same_value(ref[0], pred[0], answer_format):
        return "correct"
    if any(same_value(ref[0] * f, pred[0], answer_format) for f in POWERS_OF_1000):
        return "scale"
    return "wrong"


def normalise(value):
    text = json.dumps(value) if isinstance(value, list) else str(value)
    return " ".join(re.sub(r"[^a-z0-9%.]+", " ", text.lower()).split())


def contains_phrase(haystack, needle):
    return bool(needle) and f" {needle} " in f" {haystack} "


def canonical_item(item):
    """A list item for comparison: a number as a number ('$35,000' = 35000), anything else normalised text."""
    return f"{numbers_in(item)[0]:.6g}" if numeric_text(item) else normalise(item)


def item_set(value):
    return {canonical_item(item) for item in as_list(value)} - {""}


def items_match(a, b):
    """Equal, one a phrase of the other, or ANLS above 0.5 (S-v1's threshold)."""
    return a == b or contains_phrase(a, b) or contains_phrase(b, a) or anls_compute(a, b) > 0


# References the frozen E6 error audit labelled as annotation issues: scored by S-v1 as usual, sent to adjudication by S-v2.
e6_audit = pd.read_csv(FROZEN_DIR / "e7" / "design" / "e6_error_audit.csv")
DEBATABLE_QIDS = set(e6_audit.loc[e6_audit["category"] == "annotation issue", "qid"])


NUMERIC_REASON = {"correct": ("correct", "numeric match after normalisation (currency signs, separators, %, scale words)"),
                  "scale": ("gray", "same number up to a factor of 1,000^n (unit scale ambiguous)"),
                  "among": ("gray", "the reference value is one of several numbers in the reply"),
                  "wrong": ("wrong", "numeric mismatch")}


def score_v2(reference, prediction, answer_format, debatable=False):
    s1 = eval_score(reference, prediction, answer_format)
    is_abstention = isinstance(prediction, str) and prediction.strip().lower() == NOT_ANSWERABLE.lower()
    if answer_format == "None":
        return ("correct", "exact abstention") if s1 >= 1 else ("wrong", "answered an unanswerable question")
    if is_abstention:
        return "wrong", "false abstention"
    if s1 >= 1:
        return "correct", "S-v1 full match"
    if debatable:
        return "gray", "reference flagged as debatable by the frozen E6 audit"
    if answer_format in ("Int", "Float"):
        return NUMERIC_REASON[numeric_class(reference, prediction, answer_format) or "wrong"]
    if answer_format == "Str":
        if numeric_text(reference) and numeric_text(prediction):
            numeric = numeric_class(reference, prediction, "Float")
            if numeric in ("correct", "scale"):
                return NUMERIC_REASON[numeric]
        ref, pred = normalise(reference), normalise(prediction)
        if ref and ref == pred:
            return "correct", "equal after case and punctuation normalisation"
        if 0 < s1 < 1:
            return "gray", "partial ANLS credit (0.5 < ANLS < 1)"
        if contains_phrase(pred, ref):
            return "gray", "the reference is a phrase inside a longer reply"
        if len(pred.split()) >= 2 and contains_phrase(ref, pred):
            return "gray", "the reply is a phrase inside the reference"
        return "wrong", "string mismatch"
    # List: a reply missing a reference item is wrong (the benchmark asks for the whole list); only full matches go to a human.
    ref_items, pred_items = item_set(reference), item_set(prediction)
    if ref_items and ref_items == pred_items:
        return "correct", "same items after normalisation"
    if ref_items and all(any(items_match(r, p) for p in pred_items) for r in ref_items):
        if len(pred_items) > len(ref_items):
            return "gray", "every reference item present, with extra items"
        return "gray", "every reference item matched up to formatting"
    return "wrong", "a reference item is missing"


# Hand checks on invented values, including what S-v2 cannot settle (negation, debatable references): those stay gray or wrong for a human.
V2_CASES = [  # reference, reply, format, expected class
    ("Not answerable", NOT_ANSWERABLE, "None", "correct"), ("Not answerable", "7", "None", "wrong"), ("7", NOT_ANSWERABLE, "Int", "wrong"),
    ("1234", "1,234 units", "Int", "correct"), ("12", "12%", "Int", "correct"), ("15", "between 15 and 20", "Int", "gray"),
    ("3.5", "3.5 million", "Float", "gray"), ("1200", "$1.2 billion", "Float", "gray"), ("42.0", "17.3", "Float", "wrong"),
    ("300 million", "300", "Str", "gray"), ("Northern region", "The northern region had the most visitors", "Str", "gray"),
    ("Special Agent Ana Lee", "Ana Lee", "Str", "gray"), ("Ana Lee", "Bo Chen", "Str", "wrong"),
    ("Paris", "Not Paris but Lyon", "Str", "gray"),  # failure case: containment cannot see the negation; the adjudicator must
    ("Less well-off", "less well off", "Str", "correct"), ("Pew Research Center", "Pew Research Centre", "Str", "gray"),
    ("['Circle', 'Rectangle']", ["rectangle", "circle"], "List", "correct"), ("['$35,000', '$40,684']", [35000, 40684], "List", "correct"),
    ("['Apple Inc', 'Banana']", ["apple", "banana co"], "List", "gray"), ("['a', 'b']", ["a", "b", "c"], "List", "gray"),
    ("['a', 'b', 'c']", ["a", "b"], "List", "wrong"), ("['a']", ["x"], "List", "wrong")]
assert score_v2("12", "13", "Int", debatable=True) == ("gray", "reference flagged as debatable by the frozen E6 audit")
assert score_v2("12", "12", "Int", debatable=True)[0] == "correct"  # S-v1 = 1 stays correct
observed = {(r, str(p), f): score_v2(r, p, f)[0] for r, p, f, _ in V2_CASES}
assert all(observed[(r, str(p), f)] == expected for r, p, f, expected in V2_CASES), {k: v for k, v in observed.items()}
print(f"S-v2: {len(V2_CASES)} invented cases classified as expected (including a negation that only a human can resolve).")

S-v2: 22 invented cases classified as expected (including a negation that only a human can resolve).


In [12]:
# The frozen E6 (A, V) and E7 (V0, X4, X8) predictions on the 118 development questions, scored under S-v1 and S-v2.
ARM_FILES = {"E6 A (BM25 pages)": E6_RUN / "predictions_main_r1_A.jsonl", "E6 V (ColQwen2 pages)": E6_RUN / "predictions_main_r1_V.jsonl",
             "E7 V0 (E6 V again)": E7_RUN / "predictions_main_r1_V0.jsonl", "E7 X4": E7_RUN / "predictions_main_r1_X4.jsonl",
             "E7 X8": E7_RUN / "predictions_main_r1_X8.jsonl"}
QIDS = sorted(q for q in dev.index if dev.loc[q, "doc_id"] in HELDOUT14)
assert len(QIDS) == 118
score_rows = []
for arm, path in ARM_FILES.items():
    records = {r["qid"]: r for r in read_jsonl(path)}
    assert sorted(records) == QIDS and all(r["status"] == "ok" for r in records.values())
    for qid in QIDS:
        ref, fmt, pred = dev.loc[qid, "answer"], dev.loc[qid, "answer_format"], records[qid]["pred"]
        s1 = eval_score(ref, pred, fmt)
        v2_class, v2_reason = score_v2(ref, pred, fmt, debatable=qid in DEBATABLE_QIDS)
        score_rows.append({"arm": arm, "qid": qid, "doc_id": dev.loc[qid, "doc_id"], "answerable": bool(dev.loc[qid, "answerable"]),
                           "core_scope": bool(dev.loc[qid, "core_scope"]), "answer_format": fmt, "s1_score": s1, "s1_correct": s1 > 0,
                           "s1_strict": s1 >= 1, "v2_class": v2_class, "v2_reason": v2_reason,
                           "prediction": json.dumps(pred) if isinstance(pred, list) else str(pred)})
scores = pd.DataFrame(score_rows)

# S-v1 reproduces the frozen tables exactly.
for arm, column in (("E6 A (BM25 pages)", "score_A"), ("E6 V (ColQwen2 pages)", "score_V")):
    mine = scores[scores["arm"] == arm].set_index("qid")["s1_score"]
    assert np.allclose(mine, e6_table.loc[mine.index, column], atol=1e-12), arm
e7_table = pd.read_csv(FROZEN_DIR / "e7" / "tables" / "per_question.csv")
for arm, label in (("V0", "E7 V0 (E6 V again)"), ("X4", "E7 X4"), ("X8", "E7 X8")):
    frozen = e7_table[e7_table["arm"] == arm].set_index("qid")["score"]
    mine = scores[scores["arm"] == label].set_index("qid")["s1_score"]
    assert np.allclose(mine, frozen.loc[mine.index], atol=1e-12), arm


def summarise(frame):
    ans, una = frame[frame["answerable"]], frame[~frame["answerable"]]
    return {"S-v1 correct (score > 0), of 102": int(ans["s1_correct"].sum()), "S-v1 strict (score = 1)": int(ans["s1_strict"].sum()),
            "S-v2 automatic correct": int((ans["v2_class"] == "correct").sum()), "S-v2 gray": int((ans["v2_class"] == "gray").sum()),
            "S-v2 interval (gray wrong .. gray correct)": f"{int((ans['v2_class'] == 'correct').sum())} .. {int((ans['v2_class'] != 'wrong').sum())}",
            "S-v1 correct but S-v2 wrong": int((ans["s1_correct"] & (ans["v2_class"] == "wrong")).sum()),
            "S-v1 wrong but S-v2 correct": int((~ans["s1_correct"] & (ans["v2_class"] == "correct")).sum()),
            "correct abstentions, of 16 (both protocols)": int((una["v2_class"] == "correct").sum())}


protocol_table = pd.DataFrame({arm: summarise(frame) for arm, frame in scores.groupby("arm", sort=False)}).T
display(protocol_table)
print("Reasons for gray, all arms (answerable):")
display(scores[scores["answerable"] & (scores["v2_class"] == "gray")].groupby(["arm", "v2_reason"], sort=False).size().unstack(fill_value=0))
assert (scores[~scores["answerable"]]["v2_class"] == "correct").sum() == (scores[~scores["answerable"]]["s1_correct"]).sum()

,"S-v1 correct (score > 0), of 102",S-v1 strict (score = 1),S-v2 automatic correct,S-v2 gray,S-v2 interval (gray wrong .. gray correct),S-v1 correct but S-v2 wrong,S-v1 wrong but S-v2 correct,"correct abstentions, of 16 (both protocols)"
E6 A (BM25 pages),25,25,25,12,25 .. 37,0,0,9
E6 V (ColQwen2 pages),38,35,35,12,35 .. 47,0,0,7
E7 V0 (E6 V again),38,34,35,12,35 .. 47,0,0,7
E7 X4,31,29,29,11,29 .. 40,0,0,9
E7 X8,29,28,28,11,28 .. 39,0,0,8


Reasons for gray, all arms (answerable):


v2_reason,the reference is a phrase inside a longer reply,"every reference item present, with extra items",every reference item matched up to formatting,"same number up to a factor of 1,000^n (unit scale ambiguous)",the reply is a phrase inside the reference,reference flagged as debatable by the frozen E6 audit,the reference value is one of several numbers in the reply,partial ANLS credit (0.5 < ANLS < 1)
arm,,,,,,,,
E6 A (BM25 pages),2,2,2,2,1,2,1,0
E6 V (ColQwen2 pages),1,2,2,1,1,2,1,2
E7 V0 (E6 V again),1,1,3,1,1,2,1,2
E7 X4,3,1,3,0,1,2,0,1
E7 X8,3,2,3,0,1,2,0,0


In [13]:
# The frozen verdicts' primary differences under each protocol. Descriptive only: the frozen verdicts (S-v1) are never re-labelled.
def correct_vector(arm, rule):
    frame = scores[(scores["arm"] == arm) & scores["answerable"]].set_index("qid").sort_index()
    return {"S-v1 (frozen)": frame["s1_correct"], "S-v1 strict": frame["s1_strict"], "S-v2 lower (gray wrong)": frame["v2_class"] == "correct",
            "S-v2 upper (gray correct)": frame["v2_class"] != "wrong"}[rule].astype(int)


sensitivity_rows = {}
for label, before, after in (("E6: V - A", "E6 A (BM25 pages)", "E6 V (ColQwen2 pages)"), ("E7: X4 - V0", "E7 V0 (E6 V again)", "E7 X4"),
                             ("session noise: E7 V0 - E6 V", "E6 V (ColQwen2 pages)", "E7 V0 (E6 V again)")):
    for rule in ("S-v1 (frozen)", "S-v1 strict", "S-v2 lower (gray wrong)", "S-v2 upper (gray correct)"):
        b, a = correct_vector(before, rule), correct_vector(after, rule)
        sensitivity_rows[(label, rule)] = {"before": int(b.sum()), "after": int(a.sum()), "delta": round(float((a - b).mean()), 3),
                                           "fixed": int(((b == 0) & (a == 1)).sum()), "broken": int(((b == 1) & (a == 0)).sum())}
sensitivity_table = pd.DataFrame(sensitivity_rows).T
display(sensitivity_table)
assert sensitivity_table.loc[("E6: V - A", "S-v1 (frozen)"), ["before", "after", "fixed", "broken"]].tolist() == [25, 38, 13, 0]
assert sensitivity_table.loc[("E7: X4 - V0", "S-v1 (frozen)"), ["before", "after", "fixed", "broken"]].tolist() == [38, 31, 3, 10]

before  after  delta  fixed  broken
E6: V - A                   S-v1 (frozen)                25.0   38.0  0.127   13.0     0.0
                            S-v1 strict                  25.0   35.0  0.098   10.0     0.0
                            S-v2 lower (gray wrong)      25.0   35.0  0.098   10.0     0.0
                            S-v2 upper (gray correct)    37.0   47.0  0.098   12.0     2.0
E7: X4 - V0                 S-v1 (frozen)                38.0   31.0 -0.069    3.0    10.0
                            S-v1 strict                  34.0   29.0 -0.049    4.0     9.0
                            S-v2 lower (gray wrong)      35.0   29.0 -0.059    3.0     9.0
                            S-v2 upper (gray correct)    47.0   40.0 -0.069    3.0    10.0
session noise: E7 V0 - E6 V S-v1 (frozen)                38.0   38.0  0.000    2.0     2.0
                            S-v1 strict                  35.0   34.0 -0.010    1.0     2.0
                            S-v2 lower (gray wrong)      35.0   35.0  0.000    2.0     2.0
                            S-v2 upper (gray correct)    47.0   47.0  0.000    2.0     2.0

In [14]:
# The blind adjudication queue: each distinct (question, reply) among the gray answerable replies, once, in a seeded order, without the arm.
gray = scores[scores["answerable"] & (scores["v2_class"] == "gray")].copy()
gray["reply_key"] = gray["prediction"].map(normalise)
items = gray.groupby(["qid", "reply_key"], sort=True).agg(prediction=("prediction", "first"), reason=("v2_reason", "first"),
                                                          s1_score=("s1_score", "first"), arms=("arm", lambda a: " | ".join(sorted(a)))).reset_index()
items = items.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
items["item_id"] = [f"B1-ADJ-{i + 1:03d}" for i in range(len(items))]
queue = pd.DataFrame({"item_id": items["item_id"], "document": items["qid"].map(dev["doc_id"]),
                      "evidence_pages": items["qid"].map(lambda q: " ".join(map(str, dev.loc[q, "evidence_pages"]))),
                      "question": items["qid"].map(dev["question"]), "answer_format": items["qid"].map(dev["answer_format"]),
                      "reference": items["qid"].map(dev["answer"]), "reply": items["prediction"], "label (correct / wrong / unsure)": "", "note": ""})
key = items[["item_id", "qid", "arms", "reason", "s1_score"]]

# Human labels are durable evidence. A rerun may preserve them only when every blind item is byte-for-byte identical.
queue_path = OUT_DIR / "adjudication_queue.csv"
label_col = "label (correct / wrong / unsure)"
identity_cols = ["document", "evidence_pages", "question", "answer_format", "reference", "reply"]
preserved_labels = 0
if queue_path.exists():
    previous_queue = pd.read_csv(queue_path, dtype=str).fillna("")
    assert list(previous_queue.columns) == list(queue.columns), "Existing adjudication queue has an unexpected schema; inspect it before rerunning."
    previous_by_id = previous_queue.set_index("item_id")
    current_by_id = queue.astype(str).set_index("item_id")
    assert set(previous_by_id.index) == set(current_by_id.index), "Adjudication items changed; do not transfer human labels automatically."
    for column in identity_cols:
        mismatched = previous_by_id.index[previous_by_id[column] != current_by_id.loc[previous_by_id.index, column]]
        assert len(mismatched) == 0, f"Adjudication content changed in {column}: {mismatched.tolist()}"
    queue[label_col] = queue["item_id"].map(previous_by_id[label_col]).fillna("")
    queue["note"] = queue["item_id"].map(previous_by_id["note"]).fillna("")
    preserved_labels = int(queue[label_col].ne("").sum())
assert set(queue[label_col]).issubset({"", "correct", "wrong", "unsure"}), "Adjudication queue contains an invalid human label."
queue.to_csv(queue_path, index=False)
key.to_csv(OUT_DIR / "adjudication_key.csv", index=False)
print(f"{len(gray)} gray replies (answerable questions, all five arms) collapse to {len(queue)} distinct (question, reply) items for adjudication; preserved {preserved_labels} human labels.")
display(key.groupby("reason").size().rename("items").to_frame())

scores.drop(columns=["prediction"]).to_csv(OUT_DIR / "scoring_v1_v2.csv", index=False)
scoring_summary = {"population": "development: E6's 118 questions (102 answerable, 16 unanswerable)",
                   "per_arm": protocol_table.to_dict(orient="index"),
                   "verdict_sensitivity": {" | ".join(k): v for k, v in sensitivity_table.to_dict(orient="index").items()},
                   "adjudication_items": len(queue), "rule": "frozen verdicts use S-v1 and are never re-labelled; S-v2 is reported beside S-v1"}
(OUT_DIR / "scoring_summary.json").write_text(json.dumps(scoring_summary, indent=1, default=str))
print(f"Saved {OUT_DIR.relative_to(PROJECT_ROOT)}/scoring_v1_v2.csv (no reply text), scoring_summary.json, adjudication_queue.csv and adjudication_key.csv.")

58 gray replies (answerable questions, all five arms) collapse to 29 distinct (question, reply) items for adjudication; preserved 29 human labels.


,items
reason,
every reference item matched up to formatting,6
"every reference item present, with extra items",4
partial ANLS credit (0.5 < ANLS < 1),2
reference flagged as debatable by the frozen E6 audit,4
"same number up to a factor of 1,000^n (unit scale ambiguous)",3
the reference is a phrase inside a longer reply,7
the reference value is one of several numbers in the reply,2
the reply is a phrase inside the reference,1


Saved data/processed/mmlongbench_doc/b1/scoring_v1_v2.csv (no reply text), scoring_summary.json, adjudication_queue.csv and adjudication_key.csv.


## Validation

In [15]:
# What this execution verified (each line is an assertion above that passed), and the frozen inputs re-checked at the end.
for name in ("e5", "e6", "e7"):
    assert all(verify_sha256sums(FROZEN_DIR / name).values()), name
display(pd.Series({
    "frozen inputs": "E5, E6 and E7 equal their SHA256SUMS at start and end; nothing under experiments/frozen was written",
    "population": f"175 development questions in 20 documents; the test pool was dropped before any other read; core scope = {len(CORE_DOCS)} documents, "
                  "the same for every born-digital threshold from 0.5 to 0.84",
    "retrieval": "raw Recall@4 and Hit@4 of A, V and H reproduce E5's frozen table on 141/141 questions; E6's answer-level 13 fixed / 0 broken reproduced",
    "S-v1": "8 functions and the pattern list equal notebook 03's cell f653d232; 17 hand-traced cases; every frozen E6 (A, V) and E7 (V0, X4, X8) score reproduced",
    "S-v2": f"{len(V2_CASES)} invented cases, including a negation it cannot resolve (left gray for a human)",
    "frozen verdicts": "E6 V - A = 25 -> 38 (13 / 0) and E7 X4 - V0 = 38 -> 31 (3 / 10) under S-v1, unchanged",
}, name="verified").to_frame())

,verified
frozen inputs,"E5, E6 and E7 equal their SHA256SUMS at start and end; nothing und..."
population,175 development questions in 20 documents; the test pool was dropp...
retrieval,"raw Recall@4 and Hit@4 of A, V and H reproduce E5's frozen table o..."
S-v1,8 functions and the pattern list equal notebook 03's cell f653d232...
S-v2,"22 invented cases, including a negation it cannot resolve (left gr..."
frozen verdicts,E6 V - A = 25 -> 38 (13 / 0) and E7 X4 - V0 = 38 -> 31 (3 / 10) un...


## Results and interpretation (development data)

### A. Retrieval: ColQwen2 stays primary; no fusion is adopted

**Check.** Raw Recall@4 and Hit@4 of BM25 (A), ColQwen2 (V) and RRF (H) reproduce E5's frozen table on all 141 evidence questions.

**Complementarity is lopsided.** At B1's budget of 4 pages:

| Complete evidence at k = 4 | ColQwen2 only | BM25 only | both | neither |
|---|---:|---:|---:|---:|
| all 141 questions | 32 | 4 | 60 | 45 |
| core scope (57) | 6 | 2 | 25 | 24 |

**No combination beats ColQwen2 alone.** Complete@4 against V on the 101 held-out evidence questions:

| | difference | cluster 95 % interval |
|---|---:|---|
| RRF | −0.079 | [−0.167, −0.016] |
| Interleaving | −0.059 | [−0.133, −0.009] |
| RRF, core scope | 0.000 | [−0.064, +0.039] |

The rule stated before any number was computed is not met, so **V stays B1's primary retriever**.

**The complementarity ceiling is small.**
- Even a perfect per-question choice between BM25 and ColQwen2 adds only **+0.04** complete evidence on held-out documents, and +0.035 in the core scope.
- At the answer level, E6's reader answered **0 of 102** questions correctly only with BM25's pages (0 of 50 in the core scope).

**Where BM25 is competitive.**
- On text-dense evidence: plain text (recall@4: RRF 0.73, V 0.72, A 0.63) and financial reports (RRF 0.57, V 0.55, A 0.51).
- ColQwen2 dominates on figures (0.85 vs 0.48), layout (0.79 vs 0.38) and OCR'd pages (0.84 vs 0.52).

**What limits evidence is neither retriever.**
- For 45 of 141 questions, neither puts the complete evidence in its top 4.
- Recall falls with the evidence span: 0.91 for single-page questions, 0.60 for 2–3 pages, 0.47 for 4+ pages.
- That is the target for a later, adaptive retrieval step. Static fusion does not address it.

**Consequence for B1.** ColQwen2 primary, with BM25 as a first-class component used in four ways:
- the fallback when no visual index exists;
- its ranking logged beside ColQwen2's for every question;
- the page text used to verify citations;
- a candidate *tool* for a later adaptive step (exact terms in text-dense documents), tested only then.

### B. Scoring protocol v2: S-v1 kept; a narrow, blind human layer added

**S-v1.** It is verified unchanged against notebook 03's cell, its 17 hand-traced cases, and every frozen E6 and E7 score.

**S-v2.**
- On these 590 replies, S-v2 never contradicts S-v1 where it decides automatically.
- It sends 11–12 of 102 answerable replies per arm to adjudication, which deduplicates to **29 distinct (question, reply) items**:
  - verbose replies containing the reference: 7;
  - lists with every reference item, formatted differently or with extras: 10;
  - debatable references: 4;
  - unit-scale ambiguity: 3;
  - partial string credit: 2;
  - other: 3.

**Absolute accuracy is uncertain by about ±6 questions; paired differences are not.**

| | S-v1 (frozen) | S-v2 interval (gray wrong .. gray correct) |
|---|---:|---:|
| E6 V correct, of 102 | 38 | 35 .. 47 |
| E6: V − A | +0.127 | +0.098 at both bounds |
| E7: X4 − V0 | −0.069 | −0.059 .. −0.069 |

Both frozen verdicts keep their direction. By rule they are not re-labelled.

**Scoring uncertainty exceeds session noise.** The same request scored in two sessions flips 2 + 2 questions, while the gray zone holds about 12 per arm. Absolute accuracies therefore need the adjudication; paired comparisons on the same questions are much less affected.

### Limitations

- **Development data.** These questions were used to choose ColQwen2 (E5/E6) and to design E7.
- **A thin core scope.** It has only 6 documents; the two 10-Ks hold 31 of its 57 evidence questions.
- **Rankings without the reader.** k = 4 is applied to the rankings directly, without GPT-4o-mini's context guard, because B1's reader is not bound by it.
- **One protocol revision.** The S-v2 list rule was revised once on these replies, before the protocol is frozen. The containment rules cannot detect negation, which is shown as a failure case; a human must.

### Next step

1. **Adjudication (human, about 30 minutes).** Label `adjudication_queue.csv` (29 items) blind, following the rubric. S-v2 then becomes a single value for the E6/E7 anchors.
2. **E8 (reader migration).** It needs the owner's approval of its specification, then of the smoke stage, then of the full stage.


## C. E8 reader smoke (approved 2026-11-25)

This stage measures model-specific facts that cannot be established offline: image-token accounting, accepted request parameters, strict Structured Outputs, served-model identity, latency and list-price cost.

- `VISUALDOC_QA_MODE=replay` is the default and creates no client.
- `VISUALDOC_QA_MODE=smoke` is the only paid mode: at most seven attempts (six replies plus an optional temperature-refusal retry) and a hard cumulative ceiling of $0.10.
- S1-S5 use `mmlb-0068`, already sent in E6/E7. S6 is selected by a frozen seeded rule from answerable single-page 10-K questions whose evidence is in ColQwen2's top four.
- Requests are fingerprinted locally. Logs replace image bytes with SHA-256 values and never contain the API key.
- Smoke replies are interface diagnostics and are not scored as experimental outcomes.


In [16]:
# E8 smoke design: build and fingerprint every possible request before any API client exists.
from datetime import datetime, timezone
from io import BytesIO
from math import ceil, floor
import base64
import os
import subprocess
import time
import uuid

import openai
import tiktoken
from dotenv import load_dotenv
from PIL import Image

E8_MODE = os.getenv("VISUALDOC_QA_MODE", "replay")
assert E8_MODE in {"replay", "smoke"}, "VISUALDOC_QA_MODE must be replay or smoke"
E8_RUNS_DIR = PROJECT_ROOT / "experiments" / "runs" / "e8_smoke"
E8_MODEL = "gpt-6-luna"
E8_PRICE = {"input": 0.10 / 1e6, "cached_input": 0.01 / 1e6, "output": 0.50 / 1e6}
E8_BUDGET_USD = 0.10
E8_REASONING_EFFORT = "none"
E8_ENCODING = tiktoken.get_encoding("o200k_base")
E8_NOTEBOOK = "notebooks/07_baseline_consolidation.ipynb"
E8_SPEC = "docs/baseline_b1_specification.md"
E8_MANIFEST_PATH = PROCESSED_DIR / "e5" / "render_manifest_heldout.csv"
E8_PAGES_PATH = PROCESSED_DIR / "e5" / "heldout_pages.parquet"

V1_SYSTEM_PROMPT = """You answer questions about a single PDF document using only the page images provided.
Each page image is preceded by a tag [Page N], where N is the 1-based page number in the PDF.

Return a JSON object with exactly these keys:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the page images do not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": a list of the page numbers N whose images support the answer ([] if "Not answerable").

Do not use knowledge outside the page images. Do not guess."""

B1_SYSTEM_PROMPT = """You answer questions about a single PDF document using only the page images provided.
Each page image is preceded by a tag [Page N], where N is the 1-based page number in the PDF.

Return a JSON object with exactly these keys:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the page images do not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": a list of the page numbers N whose images support the answer ([] if "Not answerable").
  "evidence": for each cited page, one short passage copied verbatim from that page (at most 30 words)
              that supports the answer ([] if "Not answerable").

Do not use knowledge outside the page images. Do not guess."""

B1_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": ["answer", "cited_pages", "evidence"],
    "properties": {
        "answer": {"anyOf": [{"type": "string"}, {"type": "number"},
                             {"type": "array", "items": {"anyOf": [{"type": "string"}, {"type": "number"}]}}]},
        "cited_pages": {"type": "array", "items": {"type": "integer"}},
        "evidence": {"type": "array", "items": {
            "type": "object", "additionalProperties": False, "required": ["page", "quote"],
            "properties": {"page": {"type": "integer"}, "quote": {"type": "string"}}}},
    },
}
JSON_OBJECT = {"type": "json_object"}
STRICT_B1 = {"type": "json_schema", "json_schema": {"name": "b1_answer", "strict": True, "schema": B1_SCHEMA}}


def e8_header(question):
    return f"Question: {question}\n\nPage images:"


def e8_page_tag(page):
    return f"[Page {page}]"


def e8_image_part(png):
    return {"type": "image_url", "image_url": {
        "url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"), "detail": "high"}}


def e8_png_size(png):
    assert png.startswith(b"\x89PNG\r\n\x1a\n") and len(png) >= 24
    return int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")


def e8_blank_png(width, height):
    stream = BytesIO()
    Image.new("RGB", (width, height), "white").save(stream, format="PNG", optimize=True)
    data = stream.getvalue()
    assert e8_png_size(data) == (width, height)
    return data


e8_manifest = pd.read_csv(E8_MANIFEST_PATH).set_index(["doc_id", "page"])


def e8_page_png(doc_id, page):
    row = e8_manifest.loc[(doc_id, int(page))]
    png = (PROJECT_ROOT / row["image_path"]).read_bytes()
    assert sha256_bytes(png) == row["sha256"], f"stale render: {doc_id} p.{page}"
    return png


def e8_build_request(system_prompt, question, pages, response_format, max_tokens, *, temperature=None):
    content = [{"type": "text", "text": e8_header(question)}]
    for page, png in pages:
        content.extend([{"type": "text", "text": e8_page_tag(page)}, e8_image_part(png)])
    request = {
        "model": E8_MODEL,
        "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": content}],
        "response_format": response_format,
        "max_completion_tokens": max_tokens,
        "reasoning_effort": E8_REASONING_EFFORT,
    }
    if temperature is not None:
        request["temperature"] = temperature
    return request


def e8_loggable(request):
    messages = []
    for message in request["messages"]:
        raw_parts = message["content"] if isinstance(message["content"], list) else [{"type": "text", "text": message["content"]}]
        parts = []
        for part in raw_parts:
            if part["type"] == "image_url":
                png = base64.b64decode(part["image_url"]["url"].split(",", 1)[1])
                parts.append({"type": "image", "sha256": sha256_bytes(png), "bytes": len(png),
                              "dimensions": list(e8_png_size(png)), "detail": part["image_url"].get("detail")})
            else:
                parts.append(part)
        messages.append({"role": message["role"], "content": parts if isinstance(message["content"], list) else message["content"]})
    return {**{k: v for k, v in request.items() if k != "messages"}, "messages": messages}


def e8_fingerprint(request):
    return sha256_bytes(json.dumps(e8_loggable(request), sort_keys=True).encode())


def e8_image_dimensions(request):
    dimensions = []
    for message in request["messages"]:
        if isinstance(message["content"], list):
            for part in message["content"]:
                if part["type"] == "image_url":
                    dimensions.append(e8_png_size(base64.b64decode(part["image_url"]["url"].split(",", 1)[1])))
    return dimensions


def e8_legacy_image_tokens(width, height):
    scale = min(1.0, 2048 / max(width, height))
    width, height = width * scale, height * scale
    short_scale = 768 / min(width, height)
    if short_scale < 1:
        width, height = width * short_scale, height * short_scale
    return 2_833 + 5_667 * ceil(round(width, 6) / 512) * ceil(round(height, 6) / 512)


def e8_cost_upper(request):
    text_bytes = 0
    for message in request["messages"]:
        if isinstance(message["content"], str):
            text_bytes += len(message["content"].encode())
        else:
            text_bytes += sum(len(p.get("text", "").encode()) for p in message["content"] if p["type"] == "text")
    prompt_bound = text_bytes + 200 + sum(e8_legacy_image_tokens(*d) for d in e8_image_dimensions(request))
    return prompt_bound * E8_PRICE["input"] + request["max_completion_tokens"] * E8_PRICE["output"]


e8_e6_log = read_jsonl(E6_RUN / "execution_log.jsonl")
e8_e6_v = {r["qid"]: r for r in e8_e6_log if r.get("replicate_id") == "main_r1" and r.get("arm") == "V" and r.get("status") == "ok"}
S45_QID = "mmlb-0068"
assert e8_e6_v[S45_QID]["supplied_pages"] == [2, 5, 8]
s45_doc = dev.loc[S45_QID, "doc_id"]
s4_pages = [(p, e8_page_png(s45_doc, p)) for p in e8_e6_v[S45_QID]["supplied_pages"]]
s5_pages = [(p, e8_page_png(s45_doc, p)) for p in RANKING[(S45_QID, "V")][:4]]

s6_candidates = sorted(
    qid for qid, row in dev.iterrows()
    if row["doc_type"] == "Financial report" and bool(row["answerable"]) and len(row["evidence_pages"]) == 1
    and row["evidence_pages"][0] in RANKING[(qid, "V")][:4]
)
S6_QID = str(np.random.default_rng(SEED).choice(s6_candidates))
assert S6_QID == "mmlb-1026", S6_QID
s6_doc = dev.loc[S6_QID, "doc_id"]
s6_pages = [(p, e8_page_png(s6_doc, p)) for p in RANKING[(S6_QID, "V")][:4]]

portrait, landscape = e8_blank_png(768, 994), e8_blank_png(1366, 768)
question_s45, question_s6 = dev.loc[S45_QID, "question"], dev.loc[S6_QID, "question"]

E8_REQUESTS = {
    "S1": e8_build_request(B1_SYSTEM_PROMPT, question_s45, [], JSON_OBJECT, 400),
    "S2": e8_build_request(B1_SYSTEM_PROMPT, question_s45, [(1, portrait)], JSON_OBJECT, 400),
    "S3": e8_build_request(B1_SYSTEM_PROMPT, question_s45, [(1, landscape)], JSON_OBJECT, 400),
    "S4": e8_build_request(V1_SYSTEM_PROMPT, question_s45, s4_pages, JSON_OBJECT, 400, temperature=0),
    "S4prime": e8_build_request(V1_SYSTEM_PROMPT, question_s45, s4_pages, JSON_OBJECT, 400),
    "S5": e8_build_request(B1_SYSTEM_PROMPT, question_s45, s5_pages, STRICT_B1, 800),
    "S6": e8_build_request(B1_SYSTEM_PROMPT, question_s6, s6_pages, STRICT_B1, 800),
}
E8_POSSIBLE_ORDER = ["S1", "S2", "S3", "S4", "S4prime", "S5", "S6"]
E8_COST_UPPER = {label: e8_cost_upper(E8_REQUESTS[label]) for label in E8_POSSIBLE_ORDER}
assert len({e8_fingerprint(r) for r in E8_REQUESTS.values()}) == len(E8_REQUESTS)
assert sum(E8_COST_UPPER.values()) <= E8_BUDGET_USD
e8_design = {
    "model": E8_MODEL, "reasoning_effort": E8_REASONING_EFFORT, "mode_default": "replay",
    "approved_calls_max": 8, "planned_attempts_max": 7, "approved_cost_usd": E8_BUDGET_USD,
    "temperature_retry_rule": "S4prime only if S4 is refused because temperature is unsupported",
    "s45_qid": S45_QID, "s4_pages": [p for p, _ in s4_pages], "s5_pages": [p for p, _ in s5_pages],
    "s6_seed": SEED, "s6_candidate_rule": "single-page answerable Financial report; gold evidence in ColQwen2 top 4",
    "s6_qid": S6_QID, "s6_pages": [p for p, _ in s6_pages],
    "requests": {label: {"fingerprint": e8_fingerprint(request), "dimensions": e8_image_dimensions(request),
                         "cost_upper_usd": E8_COST_UPPER[label], "request": e8_loggable(request)}
                 for label, request in E8_REQUESTS.items()},
}
(OUT_DIR / "e8_smoke_design.json").write_text(json.dumps(e8_design, indent=2))
display(pd.DataFrame({
    label: {"images": len(e8_image_dimensions(E8_REQUESTS[label])), "dimensions": e8_image_dimensions(E8_REQUESTS[label]),
            "max output": E8_REQUESTS[label]["max_completion_tokens"], "temperature": E8_REQUESTS[label].get("temperature", "omitted"),
            "strict schema": E8_REQUESTS[label]["response_format"]["type"] == "json_schema",
            "conservative cost upper": E8_COST_UPPER[label]}
    for label in E8_POSSIBLE_ORDER}).T)
print(f"E8 smoke design saved offline. Mode={E8_MODE}; possible attempts={len(E8_POSSIBLE_ORDER)}; "
      f"conservative total upper=$" + f"{sum(E8_COST_UPPER.values()):.4f} <= $" + f"{E8_BUDGET_USD:.2f}.")


,images,dimensions,max output,temperature,strict schema,conservative cost upper
S1,0,[],400,omitted,False,0.000318
S2,1,"[(768, 994)]",400,omitted,False,0.002869
S3,1,"[(1366, 768)]",400,omitted,False,0.004002
S4,3,"[(768, 1087), (768, 1087), (768, 1087)]",400,0,False,0.011354
S4prime,3,"[(768, 1087), (768, 1087), (768, 1087)]",400,omitted,False,0.011354
S5,4,"[(768, 1087), (768, 1087), (768, 1087), (768, 1087)]",800,omitted,True,0.015255
S6,4,"[(768, 994), (768, 994), (768, 994), (768, 994)]",800,omitted,True,0.010718


E8 smoke design saved offline. Mode=replay; possible attempts=7; conservative total upper=$0.0559 <= $0.10.


In [17]:
# E8 smoke executor: append-only records, no SDK retries, strict budget and gate evaluation.
def e8_validate_b1(payload):
    if not isinstance(payload, dict) or set(payload) != {"answer", "cited_pages", "evidence"}:
        return False, "top-level keys differ from the strict schema"
    answer = payload["answer"]
    if not (isinstance(answer, (str, int, float, list)) and not isinstance(answer, bool)):
        return False, "answer has the wrong type"
    if isinstance(answer, list) and any(not isinstance(x, (str, int, float)) or isinstance(x, bool) for x in answer):
        return False, "an answer-list item has the wrong type"
    if not isinstance(payload["cited_pages"], list) or any(not isinstance(x, int) or isinstance(x, bool) for x in payload["cited_pages"]):
        return False, "cited_pages is not a list of integers"
    if not isinstance(payload["evidence"], list):
        return False, "evidence is not a list"
    for item in payload["evidence"]:
        if not isinstance(item, dict) or set(item) != {"page", "quote"} or not isinstance(item["page"], int) or isinstance(item["page"], bool) or not isinstance(item["quote"], str):
            return False, "an evidence item differs from the strict schema"
    return True, None


def e8_parse_reply(label, raw, finish_reason):
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError as error:
        return None, False, f"invalid JSON: {error}"
    if not isinstance(payload, dict):
        return payload, False, "reply is not a JSON object"
    valid, problem = e8_validate_b1(payload) if label in {"S5", "S6"} else (True, None)
    if finish_reason != "stop":
        return payload, False, f"finish_reason={finish_reason!r}"
    return payload, valid, problem


assert e8_validate_b1({"answer": "Not answerable", "cited_pages": [], "evidence": []}) == (True, None)
assert not e8_validate_b1({"answer": 7, "cited_pages": [1]})[0]
assert not e8_validate_b1({"answer": True, "cited_pages": [], "evidence": []})[0]


def e8_patch_count(width, height):
    patches = ceil(width / 32) * ceil(height / 32)
    if patches <= 1536:
        return patches
    scale = (1536 * 32 * 32 / (width * height)) ** 0.5
    return ceil(width * scale / 32) * ceil(height * scale / 32)


E8_TOKEN_RULES = {
    "patches_x1.2_ceil": lambda w, h: ceil(e8_patch_count(w, h) * 1.2),
    "patches_x1.2_round": lambda w, h: round(e8_patch_count(w, h) * 1.2),
    "patches_x1.2_floor": lambda w, h: floor(e8_patch_count(w, h) * 1.2),
    "patches": e8_patch_count,
    "legacy_tiles": e8_legacy_image_tokens,
}


def e8_utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def e8_append(path, payload):
    with open(path, "a") as handle:
        handle.write(json.dumps(payload, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def e8_error(error):
    message = re.sub(r"sk-[A-Za-z0-9_\-*]{4,}", "sk-<redacted>", str(getattr(error, "message", error)))
    return {"exception": type(error).__name__, "status_code": getattr(error, "status_code", None),
            "code": getattr(error, "code", None), "type": getattr(error, "type", None), "message": message[:400]}


def e8_reply_cost(record):
    prompt, cached = record.get("prompt_tokens") or 0, record.get("cached_prompt_tokens") or 0
    return max(0, prompt - cached) * E8_PRICE["input"] + cached * E8_PRICE["cached_input"] + (record.get("completion_tokens") or 0) * E8_PRICE["output"]


def e8_temperature_refusal(record):
    error = record.get("error") or {}
    return record.get("status") == "api_error" and error.get("status_code") == 400 and "temperature" in error.get("message", "").lower()


def e8_quote_observation(payload, supplied_pages, page_text):
    if not isinstance(payload, dict) or not isinstance(payload.get("evidence"), list):
        return {"quotes": 0, "pages_supplied": False, "normalised_exact_matches": 0}
    normalise_quote = lambda value: " ".join(re.sub(r"[^a-z0-9]+", " ", str(value).lower()).split())
    supplied, matches, pages_ok = set(supplied_pages), 0, True
    for item in payload["evidence"]:
        page = item.get("page")
        pages_ok &= page in supplied
        quote = normalise_quote(item.get("quote", ""))
        matches += bool(quote) and quote in normalise_quote(page_text.get(page, ""))
    return {"quotes": len(payload["evidence"]), "pages_supplied": bool(pages_ok), "normalised_exact_matches": int(matches)}


def e8_code_committed():
    return subprocess.run(["git", "diff", "--quiet", "HEAD", "--", E8_NOTEBOOK], cwd=PROJECT_ROOT).returncode == 0


def e8_spec_committed():
    return subprocess.run(["git", "diff", "--quiet", "HEAD", "--", E8_SPEC], cwd=PROJECT_ROOT).returncode == 0


e8_summary = None
if E8_MODE == "replay":
    print("E8 replay mode: no OpenAI client was created and no request was sent.")
    print("Set VISUALDOC_QA_MODE=smoke only after explicit approval; approved ceiling: seven possible attempts and $0.10.")
else:
    load_dotenv(PROJECT_ROOT / ".env")
    readiness = {
        "explicit smoke mode": E8_MODE == "smoke",
        "OPENAI_API_KEY present (value never printed)": bool(os.getenv("OPENAI_API_KEY")),
        "notebook code committed at HEAD": e8_code_committed(),
        "E8 specification committed at HEAD": e8_spec_committed(),
        "all possible attempts within approved count": len(E8_POSSIBLE_ORDER) <= 8,
        "conservative total within approved cost": sum(E8_COST_UPPER.values()) <= E8_BUDGET_USD,
    }
    display(pd.Series(readiness, name="ready").to_frame())
    if not all(readiness.values()):
        print("E8 smoke gate closed before client creation; no request was sent.")
    else:
        from openai import OpenAI

        run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid.uuid4().hex[:8]}_smoke"
        run_dir = E8_RUNS_DIR / run_id
        run_dir.mkdir(parents=True, exist_ok=False)
        log_path = run_dir / "execution_log.jsonl"
        git_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
        (run_dir / "settings.json").write_text(json.dumps({
            "run_id": run_id, "mode": E8_MODE, "model": E8_MODEL, "reasoning_effort": E8_REASONING_EFFORT,
            "approved_calls_max": 8, "planned_attempts_max": 7, "approved_budget_usd": E8_BUDGET_USD,
            "openai_sdk": openai.__version__, "git_commit": git_commit, "requests": e8_design["requests"],
        }, indent=2))
        client = OpenAI(max_retries=0, timeout=120)
        records = []

        def call_e8(label):
            request = E8_REQUESTS[label]
            record = {"label": label, "fingerprint": e8_fingerprint(request), "request": e8_loggable(request),
                      "cost_upper_usd": E8_COST_UPPER[label], "started_at": e8_utc_now(), "status": "not_attempted"}
            started = time.perf_counter()
            try:
                response = client.chat.completions.create(**request)
            except openai.OpenAIError as error:
                record.update(status="api_error", error=e8_error(error), completed_at=e8_utc_now(), latency_s=round(time.perf_counter() - started, 3))
            else:
                usage = response.usage
                details = getattr(usage, "prompt_tokens_details", None) if usage else None
                raw = response.choices[0].message.content or ""
                payload, valid, parse_error = e8_parse_reply(label, raw, response.choices[0].finish_reason)
                record.update(status="ok" if valid else "parse_error", completed_at=e8_utc_now(),
                              latency_s=round(time.perf_counter() - started, 3), served_model=response.model,
                              system_fingerprint=getattr(response, "system_fingerprint", None),
                              finish_reason=response.choices[0].finish_reason, raw_response=raw, parsed=payload,
                              parse_valid=valid, parse_error=parse_error, prompt_tokens=usage.prompt_tokens if usage else None,
                              completion_tokens=usage.completion_tokens if usage else None,
                              cached_prompt_tokens=getattr(details, "cached_tokens", None))
                record["list_price_cost_usd"] = e8_reply_cost(record)
            e8_append(log_path, record)
            records.append(record)
            return record

        assert sum(E8_COST_UPPER.values()) <= E8_BUDGET_USD
        for label in ("S1", "S2", "S3", "S4"):
            result = call_e8(label)
            if result["status"] != "ok" and label != "S4":
                break
        if records[-1]["label"] == "S4":
            result = call_e8("S4prime") if e8_temperature_refusal(records[-1]) else records[-1]
            if result["status"] == "ok":
                for label in ("S5", "S6"):
                    result = call_e8(label)
                    if result["status"] != "ok":
                        break

        reply_records = [r for r in records if r["status"] == "ok"]
        by_label = {r["label"]: r for r in reply_records}
        s4_reply_label = "S4" if "S4" in by_label else "S4prime" if "S4prime" in by_label else None
        expected_replies = {"S1", "S2", "S3", "S5", "S6"} | ({s4_reply_label} if s4_reply_label else set())
        tag_tokens = len(E8_ENCODING.encode(e8_page_tag(1)))
        measured = {}
        if {"S1", "S2", "S3"} <= set(by_label):
            measured = {(768, 994): by_label["S2"]["prompt_tokens"] - by_label["S1"]["prompt_tokens"] - tag_tokens,
                        (1366, 768): by_label["S3"]["prompt_tokens"] - by_label["S1"]["prompt_tokens"] - tag_tokens}
        matching_rules = [name for name, rule in E8_TOKEN_RULES.items()
                          if measured and all(abs(rule(*dims) - observed) <= 1 for dims, observed in measured.items())]
        served_models = {r.get("served_model") for r in reply_records}
        actual_cost = sum(e8_reply_cost(r) for r in reply_records)
        gates = {
            "six required replies received": set(by_label) == expected_replies and len(expected_replies) == 6,
            "every received reply parses and finishes with stop": all(r.get("parse_valid") and r.get("finish_reason") == "stop" for r in reply_records),
            "S5 and S6 satisfy the strict schema": all(by_label.get(label, {}).get("parse_valid") for label in ("S5", "S6")),
            "one served model string": len(served_models) == 1,
            "reasoning_effort none accepted": len(reply_records) == 6,
            "image-token rule reproduces portrait and landscape within one token": bool(matching_rules),
            "actual list-price cost <= $0.10": actual_cost <= E8_BUDGET_USD,
            "attempt count <= 8": len(records) <= 8,
        }
        held_text = pd.read_parquet(E8_PAGES_PATH).query("doc_id == @s6_doc").set_index("page")["text"].to_dict()
        observations = {
            "temperature": "accepted" if "S4" in by_label else "rejected; omitted in S4prime" if "S4prime" in by_label else "no reply",
            "served_model": next(iter(served_models)) if len(served_models) == 1 else sorted(str(x) for x in served_models),
            "system_fingerprints": sorted({str(r.get("system_fingerprint")) for r in reply_records}),
            "measured_image_tokens": {f"{w}x{h}": value for (w, h), value in measured.items()},
            "matching_token_rules": matching_rules,
            "prompt_tokens": {r["label"]: r.get("prompt_tokens") for r in reply_records},
            "completion_tokens": {r["label"]: r.get("completion_tokens") for r in reply_records},
            "latency_s": {r["label"]: r.get("latency_s") for r in reply_records},
            "S5_abstained_with_empty_evidence": bool(by_label.get("S5", {}).get("parsed", {}).get("answer") == NOT_ANSWERABLE
                                                       and by_label.get("S5", {}).get("parsed", {}).get("evidence") == []),
            "S6_quote_verification": e8_quote_observation(by_label.get("S6", {}).get("parsed"), [p for p, _ in s6_pages], held_text),
            "list_price_cost_usd": round(actual_cost, 8),
        }
        e8_summary = {"run_id": run_id, "complete": all(gates.values()), "calls_sent": len(records),
                      "reply_count": len(reply_records), "gates": gates, "observations": observations, "completed_at": e8_utc_now()}
        (run_dir / "smoke_summary.json").write_text(json.dumps(e8_summary, indent=2))
        display(pd.Series(gates, name="E8 smoke gate").to_frame())
        display(pd.Series(observations, name="measured", dtype=object).to_frame())
        print("E8 smoke outputs:", run_dir.relative_to(PROJECT_ROOT))
        print("Full E8 remains closed: this smoke approval does not authorize the 354-call run.")


E8 replay mode: no OpenAI client was created and no request was sent.
Set VISUALDOC_QA_MODE=smoke only after explicit approval; approved ceiling: seven possible attempts and $0.10.
